# MSR RQ1: how often do agent PRs violate rules from their context file?

1. Load the labelled (PR, rule) pairs
2. How many PRs fall under each rule?
3. Labels per rule
4. Violation rate, pooled and repo-averaged, overall and per agent

Rule definitions: `README.md` §2. Rule detection and checkers: `rules.py`. Hand validation: `validation.ipynb`.

In [1]:
import pandas as pd

from ess_msr.rules import load_labeled

## 1. Load the labelled data

`load_labeled()` (in `rules.py`) loads the P1 agent PRs with their context files, changed files and commit messages (`data.py`),
detects which rules each context file states, and labels every (PR, rule) pair with the checkers.

- `prs`: one row per agent PR whose repo had a context file, with `rules`, `context_hashes`, `changed_files`, `added_files`, `commit_messages`
- `pr_rule`: one row per (PR, rule) with `label` = `complied` / `violated` / `not_applicable`

In [2]:
prs, pr_rule = load_labeled()

23422 of 71677 agent PRs in 1326 repos had a context file


590 of them have no file info


## 2. How many PRs fall under each rule?

Counted before dropping PRs without file info.

In [3]:
under_rule = prs.explode("rules", ignore_index=True).dropna(subset=["rules"])

under_rule.groupby("rules").agg(prs=("id", "count"), repos=("repo", "nunique"))

,prs,repos
rules,,
changelog,917,34
commit_format,2088,144
package_manager,2002,128
required_tests,2411,173


In [4]:
pd.crosstab(under_rule.rules, under_rule.agent)

agent,Claude_Code,Copilot,Cursor,Devin,Google_Jules,OpenAI_Codex
rules,,,,,,
changelog,25,371,79,44,15,383
commit_format,210,778,276,184,0,640
package_manager,123,576,708,332,2,261
required_tests,213,1506,268,67,1,356


## 3. Labels per rule

In [5]:
pd.crosstab(pr_rule.rules, pr_rule.label)

label,complied,not_applicable,violated
rules,,,
changelog,321,381,193
commit_format,1793,0,250
package_manager,244,1712,23
required_tests,785,1287,265


## 4. Violation rate

Violation rate = violated / (complied + violated). `not_applicable` rows are left out.

`repo_averaged_rate` = mean of each repo's own rate, so a repo with 3,000 PRs counts the same as one with 10.

In [6]:
checked = pr_rule[pr_rule.label != "not_applicable"].copy()
checked["violated"] = checked.label == "violated"

summary = checked.groupby("rules").agg(
    prs=("id", "count"),
    repos=("repo", "nunique"),
    violation_rate=("violated", "mean"),
)

per_repo_rate = checked.groupby(["rules", "repo"]).violated.mean()
summary["repo_averaged_rate"] = per_repo_rate.groupby("rules").mean()
summary.round(2)

,prs,repos,violation_rate,repo_averaged_rate
rules,,,,
changelog,514,31,0.38,0.50
commit_format,2043,139,0.12,0.20
package_manager,267,61,0.09,0.18
required_tests,1050,125,0.25,0.26


In [7]:
per_agent_repo = checked.groupby(["rules", "agent", "repo"]).violated.mean()
by_agent = pd.DataFrame({
    "repo_averaged_rate": per_agent_repo.groupby(["rules", "agent"]).mean(),
    "prs": checked.groupby(["rules", "agent"]).size(),
    "repos": per_agent_repo.groupby(["rules", "agent"]).size(),
})
by_agent.round(2)

repo_averaged_rate  prs  repos
rules           agent                                       
changelog       Claude_Code                 0.70   17      5
                Copilot                     0.48  226     15
                Cursor                      0.86   52      6
                Devin                       0.98   33      2
                Google_Jules                0.29    7      1
                OpenAI_Codex                0.15  179      7
commit_format   Claude_Code                 0.05  210     39
                Copilot                     0.34  734     77
                Cursor                      0.29  275     26
                Devin                       0.31  184      5
                OpenAI_Codex                0.10  640     47
package_manager Claude_Code                 0.00   34     15
                Copilot                     0.26   80     31
                Cursor                      0.16   66     15
                Devin                       0.00   74      7
                Google_Jules                0.00    1      1
                OpenAI_Codex                0.14   12      7
required_tests  Claude_Code                 0.21  139     28
                Copilot                     0.23  647     77
                Cursor                      0.41   92     15
                Devin                       0.80   41      4
                Google_Jules                1.00    1      1
                OpenAI_Codex                0.34  130     17